# StanfordAIMI CheXagent-2-3b Test

This notebook verifies the HuggingFace `CheXagent-2-3b` model runs on Apple Silicon (`mps`).
CheXagent-2-3b is built on the InternVL2 architecture and is ~3B parameters (smaller and faster than the older 8b variant).

### Supported Prompts
Based on the model's documentation, it natively supports:
- **View Classification:** `What is the view of this chest X-ray? Options: (a) PA, (b) AP, (c) LATERAL`
- **View Matching:** `Do these images belong to the same study?`
- **Binary Disease Classification:** `Does this chest X-ray contain a {disease_name}?`
- **Disease Identification:** `Given the CXR, identify any diseases. Options: \n{disease_names}`
- **Findings Generation:** `Given the indication: "{indication}", write a structured findings section for the CXR.`
- **Section-by-Section Findings:** `Please provide a detailed description of "{anatomy}" in the chest X-ray`
- **Phrase Grounding / Detection:** `Please locate the following phrase: {phrase}` using bounding box coordinates.
- **Temporal Progression:** `Please identify the progression of {paths}: (a) worsening (b) stable (c) improving`
- **Summarization:** `Summarize the following Findings: {findings}`

In [1]:
import warnings
import logging

# --- Suppress FutureWarning from huggingface_hub (resume_download deprecated) ---
warnings.filterwarnings("ignore", category=FutureWarning, module="huggingface_hub")
warnings.filterwarnings("ignore", category=UserWarning)

# --- Suppress "Special tokens have been added" and other transformers info logs ---
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("transformers.tokenization_utils_base").setLevel(logging.ERROR)

# --- Fix: AttributeError: 'MessageFactory' object has no attribute 'GetPrototype' ---
# protobuf >= 4.21 removed MessageFactory.GetPrototype; patch it back so that
# tensorflow/grpc/sentencepiece (which still call the old API) don't spray tracebacks.
try:
    from google.protobuf import message_factory as _mf, symbol_database as _sd
    if not hasattr(_mf.MessageFactory, "GetPrototype"):
        def _get_prototype(self, descriptor):
            return _sd.Default().GetSymbol(descriptor.full_name)
        _mf.MessageFactory.GetPrototype = _get_prototype
except Exception:
    pass

In [2]:
import io
import time
import tempfile
import requests
import torch
from PIL import Image
from transformers import AutoModelForCausalLM, AutoTokenizer

# Step 1: Device setup — CheXagent-2-3b uses bfloat16
# device_map="auto" is not supported on MPS; load to CPU then move
device = "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.bfloat16
print(f"Using device: {device} | dtype: {dtype}")

Using device: mps | dtype: torch.bfloat16


In [3]:
MODEL_ID = "StanfordAIMI/CheXagent-2-3b"

# Step 2: Load Tokenizer and Model
# First run downloads ~6 GB of weights (3 safetensors shards)
_t0_load = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, trust_remote_code=True)
model = model.to(dtype).to(device)
model.eval()
model_load_s = round(time.perf_counter() - _t0_load, 2)
print("Model loaded successfully.")
print(f"  Load time: {model_load_s}s")


KeyError: 'type'

In [ ]:
# Step 3: Fetch a sample chest X-ray and save to a temp file
# tokenizer.from_list_format() expects a file path or URL, not a PIL Image
image_url = "https://upload.wikimedia.org/wikipedia/commons/3/3b/Pleural_effusion-Metastatic_breast_carcinoma_Case_166_%285477628658%29.jpg"

# Wikipedia blocks requests without a browser User-Agent
headers = {"User-Agent": "Mozilla/5.0 (compatible; research-notebook/1.0)"}
response = requests.get(image_url, headers=headers)
response.raise_for_status()
img_bytes = response.content

image = Image.open(io.BytesIO(img_bytes)).convert("RGB")

# Save to a temp file so we can pass the path to the tokenizer
tmp_file = tempfile.NamedTemporaryFile(suffix=".jpg", delete=False)
image.save(tmp_file.name)
image_path = tmp_file.name

display(image.resize((300, 300)))
print(f"Saved image to: {image_path}")

In [ ]:
# Step 4: Inference using the official CheXagent-2-3b pattern
prompt = "Describe the findings in this chest X-ray."

query = tokenizer.from_list_format([
    {'image': image_path},
    {'text': prompt},
])
conv = [
    {"from": "system", "value": "You are a helpful assistant."},
    {"from": "human",  "value": query},
]
input_ids = tokenizer.apply_chat_template(
    conv, add_generation_prompt=True, return_tensors="pt"
).to(device)

# Build attention mask (1 for all real tokens, 0 for padding)
attention_mask = torch.ones_like(input_ids)

with torch.inference_mode():
    output = model.generate(
        input_ids,
        attention_mask=attention_mask,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=False,
        num_beams=1,
        temperature=1.0,
        top_p=1.0,
        use_cache=True,
        max_new_tokens=512,
    )[0]

response = tokenizer.decode(output[input_ids.size(1):-1])
print("\n--- Response ---")
print(response)

## Local Dataset Analysis

We now extend the test to local datasets from `week1/data`. 
Based on model compatibility (Radiology domain), we include:
1. **IQ-OTH_NCCD**: Chest CT slices (Radiology - Chest).
2. **Spinal_DICOM**: Spinal CT (Radiology - Spine).

Pathology datasets (Quilt1M) are excluded as they are out-of-domain for this Chest X-ray model.

In [ ]:
from pathlib import Path
import json
import pydicom
import numpy as np

# Define paths to local samples
BASE_DATA_DIR = Path("../week1/data")

LOCAL_SAMPLES = [
    {
        "name": "IQ-OTH_NCCD (Benign)",
        "path": BASE_DATA_DIR / "IQ-OTH_NCCD/Benign/Benign_case_1.jpg",
        "type": "image",
        "prompt": "Given the indication: \"screening for lung cancer\", write a structured findings section for this chest CT slice. Focus on describing any visible nodules or lung field abnormalities."
    },
    {
        "name": "IQ-OTH_NCCD (Malignant)",
        "path": BASE_DATA_DIR / "IQ-OTH_NCCD/Malignant/Malignant_case_1.jpg",
        "type": "image",
        "prompt": "Given the indication: \"suspected pulmonary malignancy\", write a structured findings section for this chest CT slice. Identify any spiculated masses or signs of metastasis."
    },
    {
        "name": "Spinal_DICOM (Myel_001)",
        "path": BASE_DATA_DIR / "Spinal_DICOM/Myel_001/MonoE_80keVHU/1-0001.dcm",
        "type": "dicom",
        "prompt": "Given the indication: \"evaluation for multiple myeloma\", write a structured findings section for this spinal CT slice. Specifically describe the integrity of the vertebral bodies."
    },
]

def get_image_path(sample):
    p = Path(sample["path"])
    if not p.is_absolute():
        # Notebooks typically run from their own directory
        p = p.resolve()
        
    if sample["type"] == "image":
        return str(p)
    elif sample["type"] == "dicom":
        # Extract slice from DICOM and save as temporary JPEG
        if not p.exists():
            print(f"  [DEBUG] Path does not exist: {p}")
            return None
        ds = pydicom.dcmread(str(p))
        pixel_array = ds.pixel_array
        # Normalize to 0-255 for JPEG
        pixel_array = ((pixel_array - pixel_array.min()) / (pixel_array.max() - pixel_array.min()) * 255).astype(np.uint8)
        img = Image.fromarray(pixel_array).convert("RGB")
        tmp = tempfile.NamedTemporaryFile(suffix=".jpg", delete=False)
        img.save(tmp.name)
        return tmp.name
    return None

def run_inference(image_path, prompt):
    query = tokenizer.from_list_format([
        {'image': image_path},
        {'text': prompt},
    ])
    conv = [
        {"from": "system", "value": "You are a professional radiologist. Please provide a detailed and structured clinical report for the provided image."},
        {"from": "human",  "value": query},
    ]
    input_ids = tokenizer.apply_chat_template(
        conv, add_generation_prompt=True, return_tensors="pt"
    ).to(device)
    attention_mask = torch.ones_like(input_ids)

    with torch.inference_mode():
        output = model.generate(
            input_ids,
            attention_mask=attention_mask,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            num_beams=1,
            temperature=0.1,
            top_p=1.0,
            repetition_penalty=1.5,
            use_cache=True,
            max_new_tokens=512,
        )[0]

    return tokenizer.decode(output[input_ids.size(1):-1])

RESULTS = []
_t_run_start = time.perf_counter()

for sample in LOCAL_SAMPLES:
    print(f"\nProcessing: {sample['name']}")
    img_path = get_image_path(sample)
    if not img_path or not Path(img_path).exists():
        print(f"  [ERROR] Image not found: {img_path}")
        continue
    
    # Display image
    display(Image.open(img_path).resize((300, 300)))
    
    # Run model
    _t0 = time.perf_counter()
    result = run_inference(img_path, sample["prompt"])
    sample_time_s = round(time.perf_counter() - _t0, 2)
    print(f"  [{sample_time_s}s]")
    print(f"--- Response ---\n{result}")
    RESULTS.append({"sample": sample["name"], "response": result, "time_s": sample_time_s, "model_load_s": model_load_s})
total_inference_s = round(time.perf_counter() - _t_run_start, 2)
print(f"\nTotal inference time: {total_inference_s}s  |  Model load: {model_load_s}s")

CHEXAGENT_RESULTS_PATH = Path("chexagent_results.json")
with open(CHEXAGENT_RESULTS_PATH, "w") as _f:
    json.dump({"model_load_s": model_load_s, "total_inference_s": total_inference_s, "results": RESULTS}, _f, indent=2)
print(f"Results saved to {CHEXAGENT_RESULTS_PATH}")
